# Predict and Explain with NPClassifier

This notebook walks through the full inference + explainability pipeline:

1. **Load** a trained checkpoint
2. **Predict** NP-Classifier labels for SMILES strings
3. **Explain** predictions with SHAP — which Morgan fingerprint bits matter most?
4. **Draw** the important bits: highlighted on the full molecule and as isolated fragment images

> **Prerequisite:** You need a trained checkpoint. Run `01_train_from_csv.py` or `notebooks/create_dataset_and_train_models.ipynb` first.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import lightning
import matplotlib.pyplot as plt
from IPython.display import display, SVG
from torch.utils.data import DataLoader

from torch_np_classifier import (
    NPClassifierFeaturizer,
    NPClassifierDataset,
    NPClassifierLightning,
    NPClassifierSHAP,
    decode_predictions,
)

## 1. Load the trained model

Update `CKPT_PATH` to point to your checkpoint file.

In [ ]:
CKPT_PATH = "../np_classifier.ckpt"  # <-- update this path

model = NPClassifierLightning.load_from_checkpoint(CKPT_PATH)
model.eval()
print("Loaded model. hparams:", dict(model.hparams))

## 2. Predict on example molecules

We featurize a small set of well-known natural products and run inference.

In [ ]:
MOLECULES = {
    "caffeine": "Cn1cnc2c1c(=O)n(c(=O)n2C)C",
    "quercetin": "O=c1c(O)c(-c2ccc(O)c(O)c2)oc2cc(O)cc(O)c12",
    "morphine": "CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@@H]3[C@@H]1C5",
    "cholesterol": "[C@@H]1(CC[C@@H]2[C@@]1(CC[C@H]3[C@H]2CC=C4[C@@]3(CC[C@@H](C4)O)C)[C@H](CCCC(C)C)C)C",
}

featurizer = NPClassifierFeaturizer(radius=2, n_jobs=-1)
smiles_list = list(MOLECULES.values())
names_list = list(MOLECULES.keys())
features = featurizer.transform(smiles_list)  # (N, 6144)
print(f"Feature matrix: {features.shape}")

In [ ]:
dataset = NPClassifierDataset(features)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
trainer = lightning.Trainer(enable_progress_bar=False, logger=False)
raw = trainer.predict(model, loader)
probs = torch.cat(raw, dim=0).numpy()  # (N, num_categories)
print("Predictions shape:", probs.shape)

### 2.1 Decode predictions into human-readable labels

Load label names from the training CSV (columns after `SMILES` / `key`).

In [ ]:
TRAIN_CSV = "../examples/train_dataset.csv"  # <-- update if needed

df_train = pd.read_csv(TRAIN_CSV)
label_cols = [c for c in df_train.columns if c not in ("SMILES", "key")]
label_names = np.array(label_cols)
print(f"{len(label_names)} label classes loaded.")

decoded = decode_predictions(probs, label_names, threshold=0.5)
for name, dec in zip(names_list, decoded):
    print(f"\n{name}:")
    for level, preds in dec.items():
        if preds:
            print(f"  {level}: {preds}")

## 3. SHAP Explainability

We use `NPClassifierSHAP` (backed by `shap.GradientExplainer`) to attribute each prediction to individual Morgan fingerprint bits.

**Key idea:**  
Each of the 6 144 fingerprint dimensions encodes a Morgan environment (substructure centred on a specific atom at a specific radius).  
A positive SHAP value for a dimension means that substructure *increases* the predicted probability for the target class.

### 3.1 Build the background set

SHAP needs a reference (background) distribution — typically ~100 fingerprints sampled from the training set.

In [ ]:
N_BACKGROUND = 100

rng = np.random.default_rng(42)
bg_smiles_all = df_train["SMILES"].dropna().tolist()
chosen_idx = rng.choice(len(bg_smiles_all), min(N_BACKGROUND, len(bg_smiles_all)), replace=False)
bg_smiles = [bg_smiles_all[i] for i in chosen_idx]
bg_features = featurizer.transform(bg_smiles)  # (N_BACKGROUND, 6144)
print(f"Background: {bg_features.shape}")

### 3.2 Pick a molecule and its top predicted class

In [ ]:
# Choose which molecule to explain
MOL_NAME = "quercetin"

mol_idx = names_list.index(MOL_NAME)
smi = smiles_list[mol_idx]
mol_probs = probs[mol_idx]

top_class = int(np.argmax(mol_probs))
class_name = str(label_names[top_class])
print(f"Molecule : {MOL_NAME}")
print(f"Top class: {class_name!r}  (idx={top_class}, p={mol_probs[top_class]:.3f})")

### 3.3 Compute SHAP values

In [ ]:
explainer = NPClassifierSHAP(model, bg_features, class_indices=[top_class])

sv = explainer.explain_smiles(smi, featurizer)  # (1, 6144, 1)
sv_1d = sv[0, :, 0]  # (6144,)

print(f"SHAP values shape: {sv.shape}")
print(f"Non-zero SHAP values: {np.count_nonzero(sv_1d)}")
print(f"Max SHAP: {sv_1d.max():.4f}   Min SHAP: {sv_1d.min():.4f}")

### 3.4 Top important bits

In [ ]:
K = 6  # number of bits to inspect

top_indices = explainer.top_feature_indices(sv_1d, k=K, positive_only=True)
print(f"Top-{K} fingerprint indices: {top_indices}")

for rank, fp_idx in enumerate(top_indices, 1):
    from torch_np_classifier import fp_index_to_radius_bit

    env_r, bit = fp_index_to_radius_bit(fp_idx, featurizer.radius)
    print(f"  #{rank}  fp_idx={fp_idx:5d}  radius={env_r}  morgan_bit={bit:4d}  SHAP={sv_1d[fp_idx]:.4f}")

## 4. Visualisations

### 4.1 Full molecule with highlighted bits

`draw_explanation` colours the top-k Morgan environments directly on the 2-D structure. Each colour corresponds to one bit (same palette as the bar chart below).

In [ ]:
svg = explainer.draw_explanation(
    smi,
    sv_1d,
    featurizer,
    k=K,
    positive_only=True,
    size=(500, 350),
    as_svg=True,
    per_feature_colors=True,
)
display(SVG(svg))

### 4.2 Individual bit fragments

`draw_bit_fragments` calls `DrawMorganEnv` for each top SHAP bit and assembles the environments into a grid. Each cell shows the isolated substructure (centre atom + neighbourhood) that contributed most to the prediction.

In [ ]:
fig_frags = explainer.draw_bit_fragments(
    smi,
    sv_1d,
    featurizer,
    k=K,
    positive_only=True,
    fragment_size=(160, 160),
    cols=3,
)
plt.show()

### 4.3 Complete explanation figure

`explanation_figure` combines everything into a single publication-ready plot:
- **Top-left** — full molecule with bits colour-coded
- **Top-right** — horizontal bar chart of SHAP values (same colour palette)
- **Bottom** — grid of isolated bit fragments

In [ ]:
fig = explainer.explanation_figure(
    smi,
    sv_1d,
    featurizer,
    class_name=class_name,
    k=K,
    mol_size=(450, 320),
    fragment_size=(160, 160),
)
plt.show()

## 5. Explaining multiple predicted classes

Different classes may be driven by different structural features.  
Here we build **one** explainer for the top-3 predicted classes and draw a separate figure for each.

In [ ]:
N_CLASSES = 3

top_k_classes = np.argsort(mol_probs)[::-1][:N_CLASSES].tolist()
print("Top-3 predicted classes:")
for c in top_k_classes:
    print(f"  idx={c}  label={label_names[c]!r}  p={mol_probs[c]:.3f}")

# Single explainer covering all three classes at once
multi_explainer = NPClassifierSHAP(model, bg_features, class_indices=top_k_classes)
sv_multi = multi_explainer.explain_smiles(smi, featurizer)  # (1, 6144, 3)
print(f"\nMulti-class SHAP shape: {sv_multi.shape}")

In [ ]:
for class_pos, class_idx in enumerate(top_k_classes):
    cname = str(label_names[class_idx])
    sv_1d_c = sv_multi[0, :, class_pos]

    fig = multi_explainer.explanation_figure(
        smi,
        sv_1d_c,
        featurizer,
        class_name=cname,
        k=K,
    )
    plt.show()

## 6. Comparing molecules for the same class

We can explain the same class across different molecules to see which structural features are consistently important.

In [ ]:
# Pick a class that appears across multiple molecules
COMPARE_CLASS_IDX = top_class  # reuse the top class from the first analysis
COMPARE_CLASS_NAME = str(label_names[COMPARE_CLASS_IDX])

compare_explainer = NPClassifierSHAP(model, bg_features, class_indices=[COMPARE_CLASS_IDX])

print(f"Comparing molecules for class: {COMPARE_CLASS_NAME!r}\n")
for name, smi_c, prob_c in zip(names_list, smiles_list, probs):
    p = prob_c[COMPARE_CLASS_IDX]
    if p < 0.1:
        print(f"  {name}: p={p:.3f} — skipping (below threshold)")
        continue
    sv_c = compare_explainer.explain_smiles(smi_c, featurizer)
    sv_1d_c = sv_c[0, :, 0]

    fig = compare_explainer.explanation_figure(
        smi_c,
        sv_1d_c,
        featurizer,
        class_name=f"{name} — {COMPARE_CLASS_NAME}",
        k=K,
    )
    plt.show()

## 7. Saving figures

All figure functions return a `matplotlib.Figure` object. Save with `fig.savefig()`.

In [ ]:
import os

os.makedirs("explanation_outputs", exist_ok=True)

fig = explainer.explanation_figure(
    smi,
    sv_1d,
    featurizer,
    class_name=class_name,
    k=K,
)
fig.savefig(f"explanation_outputs/{MOL_NAME}.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved explanation_outputs/{MOL_NAME}.png")

# Fragment-only grid
fig_frags = explainer.draw_bit_fragments(smi, sv_1d, featurizer, k=K)
fig_frags.savefig(f"explanation_outputs/{MOL_NAME}_bits.png", dpi=150, bbox_inches="tight")
plt.close(fig_frags)
print(f"Saved explanation_outputs/{MOL_NAME}_bits.png")